# logsumexp-cross-entropy composite — cx18: stable softmax via LSE — exp(x - lse(x))

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `logsumexp-cross-entropy`, `broadcasting-rules`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "logsumexp-cross-entropy"
DD_ATOM_IDS = ["logsumexp-cross-entropy", "broadcasting-rules"]
DD_SUBTOPICS = ["Loss: logsumexp cross-entropy", "Numpy: Vectorization and broadcasting"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Stable softmax = `exp(x - lse(x))` — two atoms in one expression

1. **`logsumexp-cross-entropy`** — `logsumexp(x) = log(sum(exp(x - m))) + m`
   is the safe denominator. Subtracting it elementwise from `x` and exp'ing
   gives softmax without any explicit divide.
2. **`broadcasting-rules`** — to broadcast `lse(x)` back over the
   class axis, use `keepdim=True` so the result is `(B, 1)` instead of
   `(B,)`. Same keepdim pattern as L2 row-normalize.

Composition: `softmax(x) = exp(x - lse(x, keepdim=True))`. The keepdim on
LSE is what makes the subtract broadcast cleanly across the class axis,
and the exp-of-shifted-log eliminates the explicit divide that would
otherwise overflow.


### Composite Exercise — stable softmax via LSE — exp(x - lse(x))

**Atoms exercised together**: `logsumexp-cross-entropy`, `broadcasting-rules`

Implement `cx18_stable_softmax(logits)`. Given `logits` shape `(B, C)`,
return softmax along the class axis using the LSE identity:

1. `lse = torch.logsumexp(logits, dim=-1, keepdim=True)` → shape `(B, 1)`.
2. `out = (logits - lse).exp()` → shape `(B, C)`.

Why this works: `exp(x - log(sum(exp(x)))) = exp(x) / sum(exp(x))` — but
computed in log-space first, so the divide never happens explicitly and
huge logits don't overflow.

Constraints:
- Use `keepdim=True` on the LSE so the subtract broadcasts.
- Output must sum to 1 along the class axis (within float tol).
- Must survive logits of magnitude ~10000.


In [ ]:
def cx18_stable_softmax(logits: Tensor) -> Tensor:
    # atom: broadcasting-rules — keepdim=True gives (B, 1) so
    # the subtract broadcasts across the C axis cleanly.
    lse = t.logsumexp(logits, dim=-1, keepdim=True)
    # atom: logsumexp-cross-entropy — exp(x - lse(x)) is softmax,
    # done in log-space first so the explicit divide never happens.
    return (logits - lse).exp()


<details><summary>Show solution — cx18</summary>

```python
def cx18_stable_softmax(logits: Tensor) -> Tensor:
    # atom: broadcasting-rules — keepdim=True gives (B, 1) so
    # the subtract broadcasts across the C axis cleanly.
    lse = t.logsumexp(logits, dim=-1, keepdim=True)
    # atom: logsumexp-cross-entropy — exp(x - lse(x)) is softmax,
    # done in log-space first so the explicit divide never happens.
    return (logits - lse).exp()

```

The `keepdim=True` is the broadcasting-rules atom — same `(B, 1)`
broadcast trick as L2 row-normalize, just with `logsumexp` as the reducer.
The `exp(x - lse(x))` identity is the logsumexp-cross-entropy atom: it
rewrites `exp(x) / sum(exp(x))` in a form that never overflows.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["Loss: logsumexp cross-entropy", "Numpy: Vectorization and broadcasting"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()